# Aria Advanced Caching & Optimization Guide

**Deep dive into caching strategies, Redis integration, performance optimization, and scaling patterns.**

Learn advanced techniques for maximizing Aria's performance at scale.


## Caching Architecture

### Multi-Layer Cache Strategy

```
Request
  ↓
[Layer 1] CDN Cache (static assets, 1 hour)
  ↓
[Layer 2] Application Cache (in-memory, 5 min)
  ↓
[Layer 3] Redis Cache (distributed, 15 min)
  ↓
[Layer 4] Database Cache (query results, 1 hour)
  ↓
Database Query
```

### Cache Layers Configuration

```python
# shared/cache_manager.py
import redis
from functools import wraps
from datetime import timedelta
import json

class CacheManager:
    def __init__(self):
        # In-memory cache
        self.local_cache = {}
        self.cache_ttl = {}

        # Redis cache
        try:
            self.redis = redis.Redis.from_url(
                os.getenv("REDIS_URL", "redis://localhost:6379"),
                decode_responses=True
            )
            self.redis.ping()
        except:
            self.redis = None

    def get(self, key: str):
        """Get from cache with fallback chain."""
        # Try local cache first
        if key in self.local_cache:
            ttl = self.cache_ttl.get(key)
            if ttl and datetime.now() > ttl:
                del self.local_cache[key]
                del self.cache_ttl[key]
            else:
                return self.local_cache[key]

        # Try Redis
        if self.redis:
            try:
                value = self.redis.get(key)
                if value:
                    return json.loads(value)
            except:
                pass

        return None

    def set(self, key: str, value: any, ttl_seconds: int = 300):
        """Set cache with TTL."""
        # Local cache
        self.local_cache[key] = value
        self.cache_ttl[key] = datetime.now() + timedelta(seconds=ttl_seconds)

        # Redis
        if self.redis:
            try:
                self.redis.setex(
                    key,
                    ttl_seconds,
                    json.dumps(value)
                )
            except:
                pass

    def invalidate(self, key: str):
        """Invalidate cache entry."""
        if key in self.local_cache:
            del self.local_cache[key]

        if self.redis:
            try:
                self.redis.delete(key)
            except:
                pass

# Global cache manager
cache = CacheManager()

# Decorator for caching function results
def cached(ttl_seconds: int = 300):
    def decorator(func):
        @wraps(func)
        async def wrapper(*args, **kwargs):
            # Create cache key from function name and args
            cache_key = f"{func.__name__}:{args}:{kwargs}"

            # Check cache
            cached_value = cache.get(cache_key)
            if cached_value is not None:
                return cached_value

            # Compute and cache
            result = await func(*args, **kwargs)
            cache.set(cache_key, result, ttl_seconds)
            return result

        return wrapper
    return decorator
```

---

## Redis Integration Patterns

### Session Caching

```python
# shared/session_manager.py
import redis
import json
import uuid

class SessionManager:
    def __init__(self, redis_url: str):
        self.redis = redis.Redis.from_url(redis_url, decode_responses=True)
        self.session_ttl = 3600  # 1 hour

    def create_session(self, user_id: str) -> str:
        """Create new session."""
        session_id = str(uuid.uuid4())
        session_data = {
            "user_id": user_id,
            "created_at": datetime.now().isoformat(),
            "last_activity": datetime.now().isoformat()
        }

        self.redis.setex(
            f"session:{session_id}",
            self.session_ttl,
            json.dumps(session_data)
        )

        return session_id

    def get_session(self, session_id: str) -> dict:
        """Get session data."""
        data = self.redis.get(f"session:{session_id}")
        if data:
            return json.loads(data)
        return None

    def update_activity(self, session_id: str):
        """Update last activity timestamp."""
        data = self.get_session(session_id)
        if data:
            data["last_activity"] = datetime.now().isoformat()
            self.redis.setex(
                f"session:{session_id}",
                self.session_ttl,
                json.dumps(data)
            )
```

### Message Queue Caching

```python
# Distributed task queue with caching
class TaskQueue:
    def __init__(self, redis_url: str):
        self.redis = redis.Redis.from_url(redis_url)
        self.queue_key = "task_queue"
        self.results_key = "task_results"

    def enqueue(self, task: dict) -> str:
        """Add task to queue."""
        task_id = str(uuid.uuid4())
        task["id"] = task_id
        self.redis.rpush(self.queue_key, json.dumps(task))
        return task_id

    def dequeue(self) -> dict:
        """Get next task from queue."""
        task_json = self.redis.lpop(self.queue_key)
        return json.loads(task_json) if task_json else None

    def store_result(self, task_id: str, result: any):
        """Cache task result."""
        self.redis.setex(
            f"{self.results_key}:{task_id}",
            3600,  # 1 hour retention
            json.dumps(result)
        )

    def get_result(self, task_id: str) -> any:
        """Retrieve cached result."""
        result = self.redis.get(f"{self.results_key}:{task_id}")
        return json.loads(result) if result else None
```

### Bloom Filter for Duplicate Detection

```python
# Prevent duplicate messages/requests
from pybloom_live import BloomFilter

class DuplicateDetector:
    def __init__(self, redis_url: str):
        self.redis = redis.Redis.from_url(redis_url)
        self.bloom = BloomFilter(capacity=100000, error_rate=0.001)

    def is_duplicate(self, message_hash: str) -> bool:
        """Check if message already processed."""
        if message_hash in self.bloom:
            return True

        self.bloom.add(message_hash)
        return False

    def persist_bloom(self):
        """Save Bloom filter to Redis."""
        self.redis.setex(
            "bloom_filter",
            86400,  # 24 hours
            pickle.dumps(self.bloom)
        )
```

---

## Query Result Caching

### ORM Cache Integration

```python
# Chat message caching
@cached(ttl_seconds=1800)
async def get_user_messages(user_id: str, limit: int = 50) -> list:
    """Get cached user messages."""
    async with AsyncSession() as session:
        stmt = select(ChatMessage).where(
            ChatMessage.user_id == user_id
        ).order_by(
            ChatMessage.created_at.desc()
        ).limit(limit)

        result = await session.execute(stmt)
        return result.scalars().all()

# Invalidate on write
async def add_message(user_id: str, content: str) -> ChatMessage:
    """Add message and invalidate cache."""
    async with AsyncSession() as session:
        message = ChatMessage(user_id=user_id, content=content)
        session.add(message)
        await session.commit()

        # Invalidate cache
        cache.invalidate(f"get_user_messages:{user_id}:{50}")

        return message
```

### Semantic Search Cache

```python
# Cache embedding similarity results
class EmbeddingCache:
    def __init__(self, cache_manager: CacheManager):
        self.cache = cache_manager

    @cached(ttl_seconds=3600)
    async def find_similar_messages(
        self,
        query_embedding: list[float],
        threshold: float = 0.7
    ) -> list[str]:
        """Find similar cached messages."""
        # This would normally query Postgres pgvector
        # Here we cache the result
        pass

    def cache_embedding(self, text: str, embedding: list[float]):
        """Cache text->embedding mapping."""
        cache_key = f"embedding:{text}"
        self.cache.set(cache_key, embedding, ttl_seconds=86400)
```

---

## Performance Profiling & Optimization

### Profiling with cProfile

```python
# scripts/profile_chat_endpoint.py
import cProfile
import pstats
from io import StringIO
import asyncio
from function_app import chat_handler

async def run_profile():
    """Profile chat endpoint."""
    pr = cProfile.Profile()
    pr.enable()

    # Run handler multiple times
    for _ in range(100):
        await chat_handler({"message": "Hello"})

    pr.disable()

    # Print stats
    s = StringIO()
    ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
    ps.print_stats(20)  # Top 20 functions
    print(s.getvalue())

# Run: asyncio.run(run_profile())
```

### Line-by-Line Profiling

```python
# scripts/profile_training.py
# Requires: pip install line_profiler

from line_profiler import LineProfiler

def profile_training():
    """Profile training loop."""
    lp = LineProfiler()

    # Add functions to profile
    lp.add_function(train_model)
    lp.add_function(evaluate_model)

    # Run
    lp.enable()
    train_model("dataset.jsonl")
    lp.disable()

    # Print results
    lp.print_stats()

# Run: kernprof -l -v scripts/profile_training.py
```

### Memory Profiling

```python
# scripts/profile_memory.py
# Requires: pip install memory_profiler

from memory_profiler import profile

@profile
def load_large_dataset(path: str):
    """Memory profiled function."""
    import pandas as pd
    df = pd.read_csv(path)

    # Do processing
    result = df.groupby("category").size()

    return result

# Run: python -m memory_profiler scripts/profile_memory.py
# Output shows memory usage line by line
```

### Async Performance Optimization

```python
# Concurrent API calls with connection pooling
import aiohttp

class OptimizedAPIClient:
    def __init__(self, max_connections: int = 100):
        connector = aiohttp.TCPConnector(limit=max_connections)
        self.session = aiohttp.ClientSession(connector=connector)

    async def fetch_multiple(self, urls: list[str]) -> list:
        """Fetch multiple URLs concurrently."""
        tasks = [self.session.get(url) for url in urls]
        responses = await asyncio.gather(*tasks)
        return responses

    async def close(self):
        await self.session.close()

# Usage
async def batch_inference(prompts: list[str]):
    """Run inference on multiple prompts in parallel."""
    client = OptimizedAPIClient()

    tasks = [
        client.session.post(
            "http://localhost:8000/api/inference",
            json={"prompt": prompt}
        )
        for prompt in prompts
    ]

    results = await asyncio.gather(*tasks)
    await client.close()
    return results
```


## Scaling Patterns

### Horizontal Scaling with Load Balancing

```yaml
# Azure Load Balancer configuration
resource "azurerm_lb" "aria" {
  name                = "aria-lb"
  location            = azurerm_resource_group.main.location
  resource_group_name = azurerm_resource_group.main.name
  sku                 = "Standard"

  frontend_ip_configuration {
    name                 = "PublicIPAddress"
    public_ip_address_id = azurerm_public_ip.main.id
  }
}

resource "azurerm_lb_backend_address_pool" "aria" {
  loadbalancer_id = azurerm_lb.aria.id
  name            = "aria-backend-pool"
}

# Auto-scaling rules
resource "azurerm_monitor_autoscale_setting" "aria" {
  name                = "aria-autoscale"
  resource_group_name = azurerm_resource_group.main.name
  location            = azurerm_resource_group.main.location
  target_resource_id  = azurerm_app_service_plan.main.id

  profile {
    name = "default"

    capacity {
      default = 2
      minimum = 1
      maximum = 10
    }

    rule {
      metric_trigger {
        metric_name        = "CpuPercentage"
        metric_resource_id = azurerm_app_service_plan.main.id
        time_grain         = "PT1M"
        statistic          = "Average"
        time_window        = "PT5M"
        operator           = "GreaterThan"
        threshold          = 75
      }

      scale_action {
        direction = "Increase"
        type      = "ChangeCount"
        value     = 1
        cooldown  = "PT5M"
      }
    }
  }
}
```

### Database Replication & Sharding

```python
# Read replica routing
class DatabaseRouter:
    def __init__(self, primary_url: str, replica_urls: list[str]):
        self.primary = create_engine(primary_url)
        self.replicas = [create_engine(url) for url in replica_urls]
        self.replica_index = 0

    def get_read_connection(self):
        """Get connection for read operations."""
        conn = self.replicas[self.replica_index]
        self.replica_index = (self.replica_index + 1) % len(self.replicas)
        return conn

    def get_write_connection(self):
        """Get connection for write operations."""
        return self.primary

    async def read_query(self, query: str):
        """Execute read query on replica."""
        with self.get_read_connection() as conn:
            return await conn.execute(query)

    async def write_query(self, query: str, params: dict):
        """Execute write query on primary."""
        with self.get_write_connection() as conn:
            return await conn.execute(query, params)
```

---

## Caching Best Practices Checklist

### Strategy

- [ ] Identify hot paths (20% of requests = 80% of CPU)
- [ ] Cache at multiple layers
- [ ] Use appropriate TTLs
- [ ] Implement cache invalidation
- [ ] Monitor cache hit rates

### Implementation

- [ ] Use distributed cache (Redis) for multi-instance
- [ ] Implement cache warming for critical data
- [ ] Add cache versioning for safety
- [ ] Use cache stampede prevention (locks)
- [ ] Monitor memory usage

### Performance

- [ ] Profile before and after optimization
- [ ] Measure cache hit/miss rates
- [ ] Monitor P95 latencies
- [ ] Test with realistic load
- [ ] Track resource utilization

### Testing

- [ ] Test cache invalidation logic
- [ ] Verify consistency between layers
- [ ] Test under high concurrency
- [ ] Validate cache expiration
- [ ] Test Redis failure scenarios
